# swarm-lab en Google Colab

Este cuaderno es el **respaldo** para quien no pueda instalar `swarm-lab` en su computadora.
Hace lo mismo que en la Terminal, con dos diferencias:

- **No hay vista en vivo** (`swarm run --live`): Colab no abre ventanas. En su lugar, `--snapshot` guarda una imagen del final de la corrida.
- **Colab borra todo al cerrar la sesión.** Por eso la carpeta del equipo y los resultados se guardan en **Google Drive**, en `MyDrive/swarm-lab-work`.

Cada vez que se abre el cuaderno hay que correr las celdas 1 y 2 (conectar Drive e instalar). Lo demás se corre cuando se necesita.

## 1. Conectar Google Drive

Pide permiso para usar su Drive. Ahí queda la carpeta del equipo.

In [ ]:
import os

WORK = "/content/drive/MyDrive/swarm-lab-work"      # team folders and results live here
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:                                  # outside Colab (testing)
    WORK = os.path.abspath("swarm-lab-work")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("working folder:", os.getcwd())

## 2. Instalar swarm-lab

La primera vez en cada sesión tarda unos minutos, sobre todo PyBullet. Al final debe aparecer `[ok  ]` en Python, numpy, pybullet y matplotlib.

Si en el curso se publica una corrección, basta volver a correr esta celda: trae la versión más reciente.

In [ ]:
REPO = "/content/swarm-lab"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/ferdiex/swarm-lab.git {REPO}
else:
    !git -C {REPO} pull -q
!pip install -q -e {REPO}
!swarm check

## 3. Crear el equipo

Cambie `TEAM` por el nombre de su equipo: hasta 10 letras, números o `_`. Es el nombre que aparece en la vista y en el torneo. Si la carpeta ya existe en su Drive, no se toca.

In [ ]:
TEAM = "my_team"

if not os.path.isdir(TEAM):
    !swarm new {TEAM}
else:
    print(f"{TEAM}/ already exists in {WORK}")
!swarm check {TEAM}

## 4. Editar `team.json` y `agent.py`

En el panel de la izquierda (icono de carpeta) abra `drive/MyDrive/swarm-lab-work/` y la carpeta del equipo. Doble clic sobre `team.json` o `agent.py` abre un editor; los cambios se guardan con Ctrl+S (Cmd+S en Mac).

El manual se consulta igual que en la Terminal:

In [ ]:
!swarm man jam

In [ ]:
!swarm man

## 5. Ver una corrida (en lugar de `--live`)

Corre una partida con la semilla 1 y muestra cómo terminó. Para otra escena, agregue `--seed 7`; para jugar contra el equipo de práctica, `--rival sparring`.

In [ ]:
from IPython.display import Image, display

!swarm run {TEAM} --snapshot end.png
display(Image("end.png"))

## 6. Medir con 10 semillas

Cambie `OUT` para cada medición (por ejemplo `antes`, `relay30`, `c1_con`), para no sobrescribir las anteriores. Colab tiene 2 núcleos: con `--jobs 2` tarda unos 3 minutos sin rival y el doble con rival.

In [ ]:
OUT = "antes"
RIVAL = "none"          # none or sparring

!swarm eval {TEAM} --seeds 1-10 --jobs 2 --rival {RIVAL} --out results/{OUT}
display(Image(f"results/{OUT}/eval.png"))

## 7. Comparar mediciones

Muestra en una tabla el promedio y la desviación de todas las mediciones guardadas en `results/`.

In [ ]:
import glob, json

rows = []
for f in sorted(glob.glob("results/*/summary.json")):
    s = json.load(open(f))
    for team, d in s["summary"].items():
        rows.append((os.path.basename(os.path.dirname(f)), team, s["settings"]["rival"],
                     f"{d['in_u_mean']:.2f} +- {d['in_u_sd']:.2f}",
                     f"{d['stopped_outside_mean']:.2f} +- {d['stopped_outside_sd']:.2f}",
                     f"{d['reached_5']} of {len(s['seeds'])}"))
print(f"{'measurement':14s} {'team':12s} {'rival':9s} {'in its U':>14s} {'stopped out':>14s} {'reached 5':>10s}")
for r in rows:
    print(f"{r[0]:14s} {r[1]:12s} {r[2]:9s} {r[3]:>14s} {r[4]:>14s} {r[5]:>10s}")

## 8. Descargar la carpeta del equipo

Para la entrega del 4 de enero, o para pasarla a otra computadora: crea un zip con `team.json`, `agent.py` y, cuando exista, `fitness.py`.

In [ ]:
import zipfile

zip_path = f"{TEAM}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("team.json", "agent.py", "fitness.py"):
        if os.path.exists(f"{TEAM}/{name}"):
            z.write(f"{TEAM}/{name}")
print("created", zip_path, "with", zipfile.ZipFile(zip_path).namelist())
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass